# Traditional Baseline SoC Estimators — Panasonic 18650PF

## Objective

Establish two traditional baselines and define the comparison the ML model (Section 2) will be evaluated against:

1. **Coulomb Counting (CC)** — integrates current over time. Fails when initial SoC is unknown or sensor has offset.
2. **Extended Kalman Filter (EKF)** — model-based state estimator using the Thevenin ECM. Requires battery parameters.

**Dataset:** Panasonic 18650PF — 25°C UDDS drive cycle

**Ah-based Reference SoC:** $\\text{SoC}_{ref}(t) = \\frac{Q_{max} + \\text{Ah}(t)}{Q_{max}} \\times 100\\%$

> **Working assumptions (confirm with EEE professor):**
> - $Q_{max} = 2.9$ Ah (nominal capacity; same value used at all temperatures)
> - Battery is fully charged at the start of each drive-cycle file (per test protocol)
> - This is labelled **reference SoC**, not ground truth — both assumptions are unverified at the per-file level
> - At cold temperatures (−20°C), reference SoC correctly ends above 0% under this convention

**Battery parameters extracted from HPPC test (25°C):**
- $R_{in} = 25.48\\,\\text{m}\\Omega$
- $R_{th} = 23.98\\,\\text{m}\\Omega$
- $C_{th} = 786.85\\,\\text{F}$, $\\tau = 18.87\\,\\text{s}$


In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # non-interactive backend — works in both Jupyter and scripts
import matplotlib.pyplot as plt
import scipy.io as sio
import sys
from pathlib import Path

# Portable project root — verified using repository markers
def _find_project_root():
    """Locate this repository from its root or notebooks working directory."""
    cwd = Path.cwd().resolve()
    candidates = (cwd, cwd.parent) if cwd.name == "notebooks" else (cwd,)
    for candidate in candidates:
        if ((candidate / "AGENTS.md").is_file()
                and (candidate / "src" / "data_loader.py").is_file()):
            return candidate
    raise RuntimeError(
        "Cannot locate project repository: launch from its root or notebooks/ "
        "(requires AGENTS.md and src/data_loader.py)."
    )


PROJECT_ROOT = _find_project_root()

# Add src/ to sys.path so data_loader.py can be imported
src_path = str(PROJECT_ROOT / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from data_loader import load_panasonic_mat

# Verify key paths
INNER_ROOT = PROJECT_ROOT / 'DATASET/Panasonic/Panasonic 18650PF Data/Panasonic 18650PF Data'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'plots'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('INNER_ROOT exists:', INNER_ROOT.exists())
print('Imports OK')


PROJECT_ROOT: c:\Users\sheok\OneDrive\Desktop\New folder (2)\Major Project
INNER_ROOT exists: True
Imports OK


## 1. Load Data — 25°C UDDS Drive Cycle

The UDDS (Urban Dynamometer Driving Schedule) simulates city driving with frequent stop-and-go
current spikes — the hardest profile for Coulomb Counting.


In [2]:
udds_path = INNER_ROOT / '25degC/Drive cycles/03-21-17_00.29 25degC_UDDS_Pan18650PF.mat'
print('UDDS file exists:', udds_path.exists())

df = load_panasonic_mat(udds_path)

print(f'Shape: {df.shape}')
print(f'Duration: {df.Time_s.iloc[-1]/3600:.2f} h  ({df.Time_s.iloc[-1]:.0f} s)')
print(f'Samples: {len(df):,}')
print(f'SoC range: [{df.SoC_pct.min():.2f}%, {df.SoC_pct.max():.2f}%]')
print(f'Current range: [{df.Current_A.min():.3f} A, {df.Current_A.max():.3f} A]')
print(f'Temp range: [{df.Battery_Temp_C.min():.1f}°C, {df.Battery_Temp_C.max():.1f}°C]')
print()
print(df.head(3))


UDDS file exists: True
Shape: (224187, 7)
Duration: 6.24 h  (22448 s)
Samples: 224,187
SoC range: [6.85%, 100.00%]
Current range: [-7.529 A, 4.125 A]
Temp range: [25.6°C, 27.1°C]


### 1.1 Timestep Audit

Irregular timesteps cause integration errors. We use per-step $\Delta t$ from `diff(Time)` rather than a fixed 0.1 s.


In [3]:
dt_series = df.Time_s.diff().dropna()

n_dup = (dt_series == 0).sum()
n_gap = (dt_series > 1.0).sum()
n_neg = (dt_series < 0).sum()

print('Timestep analysis:')
print(f'  Mean dt:   {dt_series.mean():.5f} s')
print(f'  Median dt: {dt_series.median():.5f} s')
print(f'  Min dt:    {dt_series.min():.5f} s')
print(f'  Max dt:    {dt_series.max():.5f} s')
print(f'  dt=0 (duplicates):  {n_dup}')
print(f'  dt>1s (gaps):       {n_gap}')
print(f'  dt<0 (backwards):   {n_neg}')

if n_dup > 0 or n_gap > 0:
    print(f'  -> Using per-step dt with zero/negative values replaced by median')


Timestep analysis:
  Mean dt:   0.10013 s
  Median dt: 0.09899 s
  Min dt:    0.00000 s
  Max dt:    2.32300 s
  dt=0 (duplicates):  1
  dt>1s (gaps):       16
  dt<0 (backwards):   0
  -> Using per-step dt with zero/negative values replaced by median


### 1.2 Extract Working Arrays


In [4]:
time     = df.Time_s.values
voltage  = df.Voltage_V.values
current  = df.Current_A.values    # negative = discharge
temp_arr = df.Battery_Temp_C.values
soc_ref  = df.SoC_pct.values      # Ah-based Reference SoC (see assumptions above)

# Per-step dt — use actual time differences
dt_arr = np.diff(time, prepend=time[0] - np.median(np.diff(time)))
dt_median = np.median(dt_arr[dt_arr > 0])
dt_arr[dt_arr <= 0] = dt_median   # guard against zero/negative

Q_max = 2.9  # Ah

print(f'Arrays ready: {len(time):,} samples')
print(f'dt_median: {dt_median:.5f} s')
print(f'SoC: start={soc_ref[0]:.2f}%  end={soc_ref[-1]:.2f}%')


Arrays ready: 224,187 samples
dt_median: 0.09899 s
SoC: start=100.00%  end=6.85%


---
## 2. Baseline 1 — Coulomb Counting

$$\\text{SoC}(k+1) = \\text{SoC}(k) + \\frac{I(k) \\cdot \\Delta t_k}{Q_{max} \\times 3600} \\times 100$$

**Three scenarios (in order of physical realism):**

1. **Perfect CC** — correct initial SoC (100%), no sensor error. Shows theoretical ceiling; note this closely matches the Ah-based reference by construction.
2. **CC with wrong initial SoC (10% error)** — starts at 90% instead of 100%. Represents the *'phone plugged in mid-discharge'* scenario where the BMS cannot know the initial SoC.
3. **CC with 50 mA constant current bias** — systematic sensor offset. Represents a low-cost embedded current sensor drifting. Integrates linearly, not as a random walk.

> **Why not zero-mean Gaussian noise?** Integrated white noise behaves as a random walk with std
> growing as $\\sigma\\sqrt{N}\\,dt$. For 10 mA noise on a 2.9 Ah cell over 6 hours: $\\sigma_{walk} \\approx 0.004\\%$ SoC.
> This is negligible and does not constitute a meaningful robustness test.


In [5]:
def coulomb_counting(current_A, dt_s, Q_max_Ah, initial_soc_pct=100.0):
    """
    Discrete Coulomb Counting SoC estimator.

    Sign convention: current_A is NEGATIVE during discharge (McMaster dataset).
    delta_SoC = (I * dt) / (Q_max * 3600) * 100
    When I < 0 (discharge), delta_SoC < 0 (SoC decreases). Correct.
    """
    current_A = np.asarray(current_A, dtype=np.float64)
    dt_s      = np.asarray(dt_s, dtype=np.float64)
    N   = len(current_A)
    soc = np.empty(N)
    soc[0] = initial_soc_pct
    for k in range(1, N):
        delta_soc = (current_A[k-1] * dt_s[k]) / (Q_max_Ah * 3600) * 100.0
        soc[k] = np.clip(soc[k-1] + delta_soc, 0.0, 100.0)
    return soc


# Scenario 1: Perfect CC — correct initial SoC, no sensor error
# Note: this closely matches the Ah-based reference by construction (both integrate current)
soc_cc_perfect = coulomb_counting(current, dt_arr, Q_max, initial_soc_pct=100.0)

# Scenario 2: CC with wrong initial SoC (10% error)
# Represents 'phone plugged in mid-discharge' — BMS doesn't know initial SoC
WRONG_INIT_SOC = 90.0   # starting at 90% instead of true 100%
soc_cc_wrong_init = coulomb_counting(current, dt_arr, Q_max, initial_soc_pct=WRONG_INIT_SOC)

# Scenario 3: CC with 50 mA constant current bias
# Represents a sensor offset — accumulates linearly (not as random walk)
BIAS_A = 0.050   # A — constant sensor offset
current_biased = current + BIAS_A   # add constant positive offset
soc_cc_biased = coulomb_counting(current_biased, dt_arr, Q_max, initial_soc_pct=100.0)

print('Coulomb Counting done.')
print(f'  Perfect CC final SoC:       {soc_cc_perfect[-1]:.2f}%  (Ref: {soc_ref[-1]:.2f}%)')
print(f'  CC wrong init (90%) final:  {soc_cc_wrong_init[-1]:.2f}%  (Ref: {soc_ref[-1]:.2f}%)')
print(f'  CC +50mA bias final SoC:    {soc_cc_biased[-1]:.2f}%  (Ref: {soc_ref[-1]:.2f}%)')


Coulomb Counting done.
  Perfect CC final SoC:       6.98%  (Ref: 6.85%)
  CC wrong init (90%) final:  0.00%  (Ref: 6.85%)
  CC +50mA bias final SoC:    17.73%  (Ref: 6.85%)


In [6]:
def rmse(pred, true): return np.sqrt(np.mean((pred - true)**2))
def mae(pred, true):  return np.mean(np.abs(pred - true))

rmse_cc_perfect    = rmse(soc_cc_perfect,    soc_ref)
mae_cc_perfect     = mae(soc_cc_perfect,     soc_ref)
rmse_cc_wrong_init = rmse(soc_cc_wrong_init, soc_ref)
mae_cc_wrong_init  = mae(soc_cc_wrong_init,  soc_ref)
rmse_cc_biased     = rmse(soc_cc_biased,     soc_ref)
mae_cc_biased      = mae(soc_cc_biased,      soc_ref)

print('Coulomb Counting Metrics (vs Ah-based Reference SoC):')
print(f'  Perfect CC              RMSE: {rmse_cc_perfect:.4f}%  MAE: {mae_cc_perfect:.4f}%')
print(f'  CC wrong init (10% err) RMSE: {rmse_cc_wrong_init:.4f}%  MAE: {mae_cc_wrong_init:.4f}%')
print(f'  CC +50mA constant bias  RMSE: {rmse_cc_biased:.4f}%  MAE: {mae_cc_biased:.4f}%')
print()
print('Note: Perfect CC closely matches the reference by construction.')
print('      The realistic targets for ML are the wrong-init and biased scenarios.')


Coulomb Counting Metrics (vs Ah-based Reference SoC):
  Perfect CC              RMSE: 0.0838%  MAE: 0.0724%
  CC wrong init (10% err) RMSE: 9.8488%  MAE: 9.8377%
  CC +50mA constant bias  RMSE: 6.2906%  MAE: 5.4476%

Note: Perfect CC closely matches the reference by construction.
      The realistic targets for ML are the wrong-init and biased scenarios.


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)

# Top: SoC traces
axes[0].plot(time, soc_ref,           color='black',   lw=1.5, label='Ah-based Reference SoC', zorder=3)
axes[0].plot(time, soc_cc_perfect,    color='#2ca02c', lw=1.0, ls='--',
             label=f'Perfect CC  RMSE={rmse_cc_perfect:.2f}%  (≈ reference by construction)', alpha=0.85)
axes[0].plot(time, soc_cc_wrong_init, color='#d62728', lw=1.0,
             label=f'CC wrong init (90%)  RMSE={rmse_cc_wrong_init:.2f}%', alpha=0.9)
axes[0].plot(time, soc_cc_biased,     color='#ff7f0e', lw=1.0, ls=':',
             label=f'CC +50mA bias  RMSE={rmse_cc_biased:.2f}%', alpha=0.9)
axes[0].set_ylabel('SoC (%)', fontsize=11)
axes[0].set_title('Coulomb Counting Scenarios vs. Ah-based Reference — 25°C UDDS', fontsize=12, weight='bold')
axes[0].legend(fontsize=9); axes[0].grid(True, ls='--', alpha=0.5); axes[0].set_ylim(0, 115)

# Middle: absolute errors
axes[1].plot(time, np.abs(soc_cc_perfect    - soc_ref), color='#2ca02c', lw=0.8, label='|Error| Perfect CC')
axes[1].plot(time, np.abs(soc_cc_wrong_init - soc_ref), color='#d62728', lw=0.8, label='|Error| Wrong init')
axes[1].plot(time, np.abs(soc_cc_biased     - soc_ref), color='#ff7f0e', lw=0.8, ls=':', label='|Error| +50mA bias')
axes[1].set_ylabel('|SoC Error| (%)', fontsize=11)
axes[1].legend(fontsize=9); axes[1].grid(True, ls='--', alpha=0.5)

# Bottom: current
axes[2].plot(time, current, color='#1f77b4', lw=0.6)
axes[2].axhline(0, color='black', lw=0.8, ls=':')
axes[2].set_ylabel('Current (A)', fontsize=11)
axes[2].set_xlabel('Time (s)', fontsize=11)
axes[2].grid(True, ls='--', alpha=0.5)

plt.tight_layout()
out = RESULTS_DIR / '02_coulomb_counting_UDDS.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.close()
print(f'Saved: {out}')


---
## 3. Baseline 2 — Extended Kalman Filter (EKF)

**State vector:** $\mathbf{x}(k) = [\text{SoC}(k),\; V_{th}(k)]^\top$

**Predict:**
$$\text{SoC}(k{+}1) = \text{SoC}(k) + \frac{\eta I(k)\Delta t}{Q_{max}\times 3600}\times 100, \quad V_{th}(k{+}1) = e^{-\Delta t/\tau}V_{th}(k) + R_{th}(1-e^{-\Delta t/\tau})I(k)$$

**Update:**
$$V_{pred} = V_{OC}(\text{SoC}) + I\cdot R_{in} + V_{th}, \quad K = PH^\top(HPH^\top+R)^{-1}, \quad \mathbf{x} \leftarrow \mathbf{x} + K(V_{meas}-V_{pred})$$


**Current convention:** Negative current is discharge. The positive current term
therefore lowers SoC during discharge. The signed polarization state becomes
negative during discharge; both $I R_{in}$ and $V_{th}$ then lower terminal voltage.

**T005A interpretation and historical evidence:** This experiment initializes
`x0 = [soc_ref[0], 0]`, using the reference initial SoC, and tracks the full UDDS
trajectory. It is not directly comparable to the selected ML estimator's unknown
initial-SoC task on a short observation window. Saved EKF outputs, metrics, plots,
and comparison tables are historical and were not recomputed after this sign
repair. Focused prediction tests do not validate full-trajectory EKF accuracy.


**T005B time alignment:** `x0` and `P0` represent `time[0]`. At sample 0,
perform only the voltage measurement update; do not propagate or add process noise.
For `k >= 1`, propagate over `dt_arr[k]` (the interval `time[k-1]` to `time[k]`)
with `current[k-1]`, matching left-endpoint Coulomb counting. Use `current[k]`
for the ohmic term in the voltage update at sample k. Negative current remains
discharge. This convention assumes valid increasing timestamps; the existing
upstream replacement of nonpositive time differences is unchanged.

All saved EKF outputs predate the T005A/T005B correctness repairs. They have not
been recomputed and must not be quoted as results of the repaired loop.
Deterministic synthetic loop checks do not establish experimental accuracy.


In [8]:
# ── Load OCV-SoC lookup from C/20 test ───────────────────────────────────────
# The C/20 terminal voltage is used as a QUASI-OCV APPROXIMATION.
# At C/20 rate (0.145 A): ohmic drop ≈ 3.7 mV, RC polarisation ≈ 3.5 mV.
# Total systematic offset ≈ 7.2 mV over a 1.7 V range — acceptable (<0.5%).
# True equilibrium OCV requires extended rest periods not present in this dataset.
c20_path = (INNER_ROOT / '25degC'
             / 'C20 OCV and 1C discharge tests_start_of_tests'
             / '05-08-17_13.26 C20 OCV Test_C20_25dC.mat')
print('C/20 file exists:', c20_path.exists())

mat_c20  = sio.loadmat(c20_path)
meas_c20 = mat_c20['meas']

time_c20 = meas_c20['Time'][0, 0].flatten()
volt_c20 = meas_c20['Voltage'][0, 0].flatten()
curr_c20 = meas_c20['Current'][0, 0].flatten()
ah_c20   = meas_c20['Ah'][0, 0].flatten()

# Identify the main discharge segment
mask_dis = curr_c20 < -0.1

# CORRECTED Q_c20: use actual Ah delivered in the discharge segment (segment delta),
# NOT abs(Ah.min()). The Ah counter is not at zero at the start of the discharge
# segment (a ~0.027 Ah offset exists from a prior small discharge).
ah_dis_start = ah_c20[mask_dis][0]    # Ah at first discharge sample
ah_dis_end   = ah_c20[mask_dis][-1]   # Ah at last discharge sample
Q_c20 = abs(ah_dis_end - ah_dis_start) # ≈ 2.9949 Ah (actual segment capacity)

ah_dis   = ah_c20[mask_dis]
volt_dis = volt_c20[mask_dis]

# SoC axis: 100% at start of C/20 discharge → 0% at end
# Using relative Ah from segment start, normalised by segment capacity
soc_ocv  = np.clip((ah_dis - ah_dis_start) / (ah_dis_end - ah_dis_start) * (-100.0) + 100.0,
                   0.0, 100.0)

# Sort ascending SoC for np.interp
sort_idx        = np.argsort(soc_ocv)
soc_ocv_sorted  = soc_ocv[sort_idx]
volt_ocv_sorted = volt_dis[sort_idx]

def ocv_lookup(soc_pct):
    return np.interp(np.clip(soc_pct, 0.0, 100.0), soc_ocv_sorted, volt_ocv_sorted)

print(f'Quasi-OCV table: {len(soc_ocv_sorted)} points, Q_c20={Q_c20:.4f} Ah (corrected segment delta)')
for s in [100, 80, 50, 20, 5]:
    print(f'  OCV({s:3d}%) = {ocv_lookup(s):.4f} V')


C/20 file exists: True
Quasi-OCV table: 1241 points, Q_c20=2.9949 Ah (corrected segment delta)
  OCV(100%) = 4.1703 V
  OCV( 80%) = 3.9458 V
  OCV( 50%) = 3.6654 V
  OCV( 20%) = 3.4610 V
  OCV(  5%) = 3.2560 V


In [9]:
# ── EKF Parameters (from HPPC at 25°C) ──────────────────────────────────────
R_IN  = 0.02548   # Ω — ohmic resistance (HPPC instantaneous voltage drop)
R_TH  = 0.02398   # Ω — RC resistance (HPPC relaxation fit)
C_TH  = 786.85    # F — RC capacitance
TAU   = R_TH * C_TH
ETA   = 1.0       # Coulombic efficiency (1.0 for discharge)
Q_MAX = Q_max     # 2.9 Ah

# ── Principled noise covariance selection ─────────────────────────────────────
# Q_SoC: derived from maximum SoC rate of change
#   max|dSoC/dt| = I_max * dt / (Q_max * 3600) * 100
#                = 7.5 A * 0.1 s / (2.9 Ah * 3600) * 100 = 0.0072% per step
#   Q_SoC ≈ (0.0072)^2 = 5e-5  (variance of per-step SoC uncertainty)
# Q_Vth: RC voltage changes more slowly; use 1e-6 as before
Q_noise = np.diag([5e-5, 1e-6])

# R_noise: voltage measurement noise variance
#   Digatron accuracy ≈ 0.1% of 18V full scale = 18 mV → σ_V ≈ 10 mV
#   R ≈ (0.010)^2 = 1e-4 V^2
R_noise = np.array([[1e-4]])

# Initial state: SoC from reference (assumes fully charged start), V_th = 0 (at rest)
x0 = np.array([soc_ref[0], 0.0])
P0 = np.diag([1.0, 1e-4])

print(f'EKF params: R_in={R_IN*1e3:.2f}mΩ  R_th={R_TH*1e3:.2f}mΩ  C_th={C_TH:.1f}F  tau={TAU:.2f}s')
print(f'Q_noise = diag({np.diag(Q_noise)})  (principled: from max dSoC/dt and RC dynamics)')
print(f'R_noise = {R_noise.flatten()}  (principled: from Digatron 0.1%/18V accuracy → σ≈10mV)')


EKF params: R_in=25.48mΩ  R_th=23.98mΩ  C_th=786.9F  tau=18.87s
Q_noise = diag([5.e-05 1.e-06])  (principled: from max dSoC/dt and RC dynamics)
R_noise = [0.0001]  (principled: from Digatron 0.1%/18V accuracy → σ≈10mV)


In [10]:
def ekf_predict_state(x, I, dt, ETA, Q_MAX, R_TH, TAU):
    """Predict [SoC percent, signed polarization voltage]; I < 0 discharges.

    Assumes finite physical inputs, dt >= 0, Q_MAX > 0 and TAU > 0.
    Returns the predicted state and RC decay for the covariance prediction.
    """
    alpha = np.exp(-dt / TAU)
    soc_pred = float(np.clip(x[0] + (ETA * I * dt) / (Q_MAX * 3600) * 100.0, 0.0, 100.0))
    vth_pred = alpha * x[1] + R_TH * (1.0 - alpha) * I
    return np.array([soc_pred, vth_pred]), alpha


# ── EKF Main Loop ─────────────────────────────────────────────────────────────
N = len(time)
soc_ekf = np.empty(N)
vth_ekf = np.empty(N)

x = x0.copy()
P = P0.copy()

for k in range(N):
    I = float(current[k])    # measurement current at sample k; negative = discharge
    V_meas = float(voltage[k])

    # ── PREDICT ──────────────────────────────────────────────────────────────
    if k == 0:
        # x0 and P0 already describe time[0]; no elapsed interval or process noise.
        x_pred = x.copy()
        P_pred = P.copy()
    else:
        dt = float(dt_arr[k])  # interval time[k-1] -> time[k]
        I_prop = float(current[k-1])  # left endpoint, matching Coulomb counting
        x_pred, alpha = ekf_predict_state(x, I_prop, dt, ETA, Q_MAX, R_TH, TAU)

        F = np.array([[1.0, 0.0], [0.0, alpha]])
        P_pred = F @ P @ F.T + Q_noise

    # ── UPDATE ───────────────────────────────────────────────────────────────
    V_pred = ocv_lookup(x_pred[0]) + I * R_IN + x_pred[1]
    innov  = V_meas - V_pred

    # Linearized measurement Jacobian (numerical derivative of OCV)
    d_soc     = 0.5
    dVoc_dSoC = (ocv_lookup(x_pred[0] + d_soc) - ocv_lookup(x_pred[0] - d_soc)) / (2.0 * d_soc)
    H = np.array([[dVoc_dSoC, 1.0]])   # (1, 2)

    S = H @ P_pred @ H.T + R_noise             # (1, 1)
    K = P_pred @ H.T @ np.linalg.inv(S)        # (2, 1)

    x = x_pred + K.flatten() * innov
    x[0] = np.clip(x[0], 0.0, 100.0)

    # Covariance update — Joseph form (numerically stable)
    IKH = np.eye(2) - K @ H
    P   = IKH @ P_pred @ IKH.T + K @ R_noise @ K.T

    soc_ekf[k] = x[0]
    vth_ekf[k] = x[1]

rmse_ekf = rmse(soc_ekf, soc_ref)
mae_ekf  = mae(soc_ekf,  soc_ref)
print(f'EKF done.  RMSE={rmse_ekf:.4f}%  MAE={mae_ekf:.4f}%')
print(f'Final SoC — True: {soc_ref[-1]:.2f}%  EKF: {soc_ekf[-1]:.2f}%')


EKF done.  RMSE=2.4021%  MAE=2.1485%
Final SoC — True: 6.85%  EKF: 9.87%


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)

axes[0].plot(time, soc_ref,    color='black',   lw=1.5, label='Ah-based Reference SoC', zorder=3)
axes[0].plot(time, soc_ekf,     color='#9467bd', lw=1.2, label=f'EKF  RMSE={rmse_ekf:.2f}%')
axes[0].plot(time, soc_cc_wrong_init, color='#d62728', lw=0.7, ls=':',  alpha=0.6,
             label=f'CC wrong init  RMSE={rmse_cc_wrong_init:.2f}% [ref]')
axes[0].set_ylabel('SoC (%)', fontsize=11)
axes[0].set_title('EKF vs. Coulomb Counting vs. Ah-based Reference SoC — 25°C UDDS', fontsize=13, weight='bold')
axes[0].legend(fontsize=10); axes[0].grid(True, ls='--', alpha=0.5); axes[0].set_ylim(0, 110)

axes[1].plot(time, np.abs(soc_ekf     - soc_ref), color='#9467bd', lw=0.8, label='|Error| EKF')
axes[1].plot(time, np.abs(soc_cc_wrong_init - soc_ref), color='#d62728', lw=0.7, ls=':', alpha=0.7,
             label='|Error| CC wrong init')
axes[1].set_ylabel('|SoC Error| (%)', fontsize=11)
axes[1].legend(fontsize=10); axes[1].grid(True, ls='--', alpha=0.5)

axes[2].plot(time, vth_ekf * 1000, color='#ff7f0e', lw=0.8)
axes[2].set_ylabel('$V_{th}$ (mV)', fontsize=11)
axes[2].set_xlabel('Time (s)', fontsize=11)
axes[2].set_title('EKF Internal State: Polarization Voltage $V_{th}(t)$', fontsize=10)
axes[2].grid(True, ls='--', alpha=0.5)

plt.tight_layout()
out = RESULTS_DIR / '02_ekf_UDDS.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.close()
print(f'Saved: {out}')


---
## 4. Baseline Comparison Table

The ML model will be evaluated against **CC with realistic initialisation error** — not just against the untuned EKF.
The EKF is configured with principled Q and R values derived from physical arguments (not tuned on the test set).


In [13]:
results = {
    'Method':   [
        'CC (perfect, theoretical ceiling)',
        'CC with 10% wrong initial SoC',
        'CC with 50 mA constant bias',
        'EKF (principled Q/R, HPPC params)',
        'Target ML (Section 2)',
    ],
    'RMSE (%)': [
        f'{rmse_cc_perfect:.4f}',
        f'{rmse_cc_wrong_init:.4f}',
        f'{rmse_cc_biased:.4f}',
        f'{rmse_ekf:.4f}',
        'TBD',
    ],
    'MAE (%)':  [
        f'{mae_cc_perfect:.4f}',
        f'{mae_cc_wrong_init:.4f}',
        f'{mae_cc_biased:.4f}',
        f'{mae_ekf:.4f}',
        'TBD',
    ],
    'Notes': [
        'Theoretical ceiling; ≈ reference by construction — not a real-world baseline',
        'Primary ML target: phone plugged in mid-discharge',
        'Sensor offset: accumulates linearly at 10.7% over 6 hours',
        'Principled EKF: Q=diag(5e-5, 1e-6), R=1e-4; requires battery model params',
        'Goal: RMSE < CC-wrong-init without needing initial SoC or model params',
    ]
}
df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

# Save to CSV
out_csv = PROJECT_ROOT / 'results' / 'baseline_metrics.csv'
df_results.to_csv(out_csv, index=False)
print(f'\nSaved metrics: {out_csv}')


                           Method RMSE (%) MAE (%)                                                                        Notes
CC (perfect, theoretical ceiling)   0.0838  0.0724 Theoretical ceiling; ≈ reference by construction — not a real-world baseline
    CC with 10% wrong initial SoC   9.8488  9.8377                            Primary ML target: phone plugged in mid-discharge
      CC with 50 mA constant bias   6.2906  5.4476                    Sensor offset: accumulates linearly at 10.7% over 6 hours
EKF (principled Q/R, HPPC params)   2.4021  2.1485    Principled EKF: Q=diag(5e-5, 1e-6), R=1e-4; requires battery model params
            Target ML (Section 2)      TBD     TBD       Goal: RMSE < CC-wrong-init without needing initial SoC or model params

Saved metrics: c:\Users\sheok\OneDrive\Desktop\New folder (2)\Major Project\results\baseline_metrics.csv


---
## 5. Summary & Key Observations

| Finding | Implication |
| :--- | :--- |
| Perfect CC ≈ Ah-based reference by construction | Both integrate current; not a real-world test |
| CC with 10% wrong initial SoC → ~10% RMSE | Primary ML target: a model without initial SoC knowledge must do better |
| 50 mA constant bias accumulates to ~10.7% at 6 hours | Sensor quality matters; ML uses the same noisy V, I, T |
| EKF with principled Q/R: see result above | Needs R_in, R_th, C_th — which change with temperature and aging |
| EKF OCV table only valid at 25°C | At −10°C/−20°C, same SoC gives lower voltage → EKF degrades |
| ML target: RMSE < CC with realistic init error | V, I, T as inputs — no initial SoC needed, no battery model params |

**Working scientific question:**
> Can a recurrent neural network trained on multi-temperature Panasonic 18650PF drive-cycle data
> estimate the Ah-based reference SoC using only terminal voltage, current, and battery temperature,
> at an operating temperature unseen during training, and how does its accuracy compare with
> Coulomb Counting (subject to realistic initialisation uncertainty) and a principled EKF?

**Next step → Section 2:** Feature engineering + sliding-window dataset for LSTM training.
